In [1]:
import pandas as pd
import requests
import time

In [2]:
try:
    fact_df = pd.read_csv('../data/processed/fact_game_statistics.csv')
except FileNotFoundError:
    print("fact table not found")
    exit()

A Steam API-ból a top 200 legnagyobb játékosbázissal rendelkező játékhoz lekérdezem az átlagos achievement teljesítettségüket, és egy külön táblában tárolom ezt.

In [3]:
games_to_fetch = fact_df.sort_values("peak_ccu", ascending=False).head(200).copy()

ach_data_list = []

for app_id in games_to_fetch["game_id"]:
    url = f"https://api.steampowered.com/ISteamUserStats/GetGlobalAchievementPercentagesForApp/v0002/?gameid={app_id}"
    avg_completion_pct = 0.0
    
    try:
        response = requests.get(url, timeout=10)
        if response.status_code == 200:
            data = response.json()
            
            ach_data = data.get('achievementpercentages', {})
            
            if isinstance(ach_data, dict):
                achievements = ach_data.get('achievements', [])
                
                if len(achievements) > 0:
                    total_percent = 0.0
                    for ach in achievements:
                        try:
                            total_percent += float(ach.get('percent', 0.0))
                        except (ValueError, TypeError):
                            pass
                        
                    avg_completion_pct = round(total_percent / len(achievements), 2)
                    
        elif response.status_code == 429:
            print(f"Too many requests")
            break
        else:
            print(f"Failed to fetch data for app_id {app_id}, status code: {response.status_code}")
            
    except Exception as e:
        print(f"Error fetching data for app_id {app_id}: {e}")
        avg_completion_pct = 0.0
    
    ach_data_list.append({
        "game_id": app_id,
        "avg_ach_pct": avg_completion_pct
    })
    time.sleep(0.5) 
print("done") 

Failed to fetch data for app_id 1671210, status code: 403
Failed to fetch data for app_id 438100, status code: 403
Failed to fetch data for app_id 3241660, status code: 403
Failed to fetch data for app_id 322330, status code: 403
Failed to fetch data for app_id 1222670, status code: 403
Failed to fetch data for app_id 108600, status code: 403
Failed to fetch data for app_id 39210, status code: 403
Failed to fetch data for app_id 294100, status code: 403
Failed to fetch data for app_id 2694490, status code: 403
Failed to fetch data for app_id 306130, status code: 403
Failed to fetch data for app_id 629520, status code: 403
Failed to fetch data for app_id 1665460, status code: 403
Failed to fetch data for app_id 1049590, status code: 403
Failed to fetch data for app_id 365670, status code: 403
Failed to fetch data for app_id 1283970, status code: 403
Failed to fetch data for app_id 216150, status code: 403
Failed to fetch data for app_id 2479810, status code: 403
Failed to fetch data for

A 403-mas hibakód azt jelenti, hogy az API megtagadja a válaszadást.

In [4]:
fact_achievements = pd.DataFrame(ach_data_list)

fact_achievements.to_csv("../data/processed/fact_ach_pct.csv", index=False)